In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE DATABASE YOUR_DB').collect()
session.sql('USE SCHEMA PUBLIC').collect()

In [ ]:
df = session.table('YOUR_DB.PUBLIC.RADIOLOGY_APPOINTMENTS').to_pandas()
df

In [ ]:
from prophet import Prophet
import pandas as pd
import numpy as np

prophet_df = df.rename(columns={'DATE': 'ds', 'APPOINTMENTS': 'y'})
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])
prophet_df = prophet_df.sort_values('ds').reset_index(drop=True)
print(f"Training data: {prophet_df['ds'].min().date()} to {prophet_df['ds'].max().date()}")
print(f"Total rows: {len(prophet_df)}")
prophet_df.tail()

In [ ]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
model.add_country_holidays(country_name='US')
model.fit(prophet_df)

In [ ]:
future = model.make_future_dataframe(periods=365, freq='D')
forecast = model.predict(future)

last_date = prophet_df['ds'].max()
next_day = forecast[forecast['ds'] == last_date + pd.Timedelta(days=1)][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
next_week = forecast[(forecast['ds'] > last_date) & (forecast['ds'] <= last_date + pd.Timedelta(days=7))]
next_month = forecast[(forecast['ds'] > last_date) & (forecast['ds'] <= last_date + pd.Timedelta(days=30))]
next_year = forecast[(forecast['ds'] > last_date) & (forecast['ds'] <= last_date + pd.Timedelta(days=365))]

print("=== NEXT DAY ===")
print(f"Date: {next_day['ds'].values[0]}")
print(f"Predicted appointments: {next_day['yhat'].values[0]:.0f} (range: {next_day['yhat_lower'].values[0]:.0f} - {next_day['yhat_upper'].values[0]:.0f})")

print("\n=== NEXT WEEK (7 days) ===")
print(f"Total predicted appointments: {next_week['yhat'].sum():.0f}")
print(f"Daily average: {next_week['yhat'].mean():.0f}")

print("\n=== NEXT MONTH (30 days) ===")
print(f"Total predicted appointments: {next_month['yhat'].sum():.0f}")
print(f"Daily average: {next_month['yhat'].mean():.0f}")

print("\n=== NEXT YEAR (365 days) ===")
print(f"Total predicted appointments: {next_year['yhat'].sum():.0f}")
print(f"Daily average: {next_year['yhat'].mean():.0f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(prophet_df['ds'], prophet_df['y'], 'k.', markersize=2, label='Actual')
axes[0].plot(forecast['ds'], forecast['yhat'], color='blue', label='Forecast')
axes[0].fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'], alpha=0.2, color='blue')
axes[0].axvline(x=last_date, color='red', linestyle='--', label='Forecast start')
axes[0].set_title('Radiology Appointments - Full Forecast')
axes[0].set_ylabel('Appointments')
axes[0].legend()

next_30 = forecast[(forecast['ds'] > last_date) & (forecast['ds'] <= last_date + pd.Timedelta(days=30))]
axes[1].plot(next_30['ds'], next_30['yhat'], 'b-o', markersize=4, label='Forecast')
axes[1].fill_between(next_30['ds'], next_30['yhat_lower'], next_30['yhat_upper'], alpha=0.2, color='blue')
for _, row in next_30.iterrows():
    if row['ds'].dayofweek == 0:
        axes[1].annotate(f"{row['yhat']:.0f}", (row['ds'], row['yhat']), textcoords='offset points', xytext=(0, 10), fontsize=7, ha='center')
axes[1].set_title('Next 30 Days - Detailed Forecast')
axes[1].set_ylabel('Appointments')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
future_only = forecast[forecast['ds'] > last_date].copy()
future_only['month'] = future_only['ds'].dt.to_period('M')
monthly = future_only.groupby('month').agg(
    total_appts=('yhat', 'sum'),
    avg_daily=('yhat', 'mean'),
    lower=('yhat_lower', 'sum'),
    upper=('yhat_upper', 'sum')
).head(12).reset_index()
monthly['month_label'] = monthly['month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(monthly['month_label'], monthly['total_appts'], color='steelblue', edgecolor='white')
ax.errorbar(monthly['month_label'], monthly['total_appts'],
            yerr=[monthly['total_appts'] - monthly['lower'], monthly['upper'] - monthly['total_appts']],
            fmt='none', color='gray', capsize=4, alpha=0.5)
for bar, val in zip(bars, monthly['total_appts']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{val:,.0f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Forecasted Monthly Radiology Appointments (Next 12 Months)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Total Appointments')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(monthly[['month_label', 'total_appts', 'avg_daily']].to_string(index=False))